In [1]:
import sys
sys.path.append("..")         

from src import (
    load_raw, split_data,
    build_logreg, build_boosting, evaluate, oof_proba,
    optimal_threshold, theoretical_threshold,
)

df = load_raw()
X_train, X_test, y_train, y_test = split_data(df)

In [2]:
for name, model in [("логрег ", build_logreg()), ("бустинг", build_boosting())]:
    r = evaluate(model, X_train, y_train)
    print(f"{name}: {r['mean']:.4f} ± {r['std']:.4f}")

логрег : 0.8133 ± 0.0034
бустинг: 0.8636 ± 0.0024


In [3]:
proba = oof_proba(build_boosting(), X_train, y_train)
opt = optimal_threshold(y_train, proba)

print(f"порог           : {opt['threshold']:.3f}  (теория {theoretical_threshold():.3f})")
print(f"одобрено        : {opt['approval_rate']:.1%}")
print(f"прирост прибыли : {opt['uplift']:.1%}")

порог           : 0.164  (теория 0.167)
одобрено        : 89.4%
прирост прибыли : 19.9%


In [4]:
configs = {
    "без обработки": dict(mask_codes=False, add_flags=False, add_total=False),
    "только флаги":  dict(mask_codes=False, add_flags=True,  add_total=False),
    "маскирование":  dict(mask_codes=True,  add_flags=False, add_total=False),
    "всё вместе":    dict(),
}
for name, kw in configs.items():
    r = evaluate(build_logreg(**kw), X_train, y_train)
    print(f"{name:16s} {r['mean']:.4f} ± {r['std']:.4f}")

без обработки    0.6936 ± 0.0096
только флаги     0.8125 ± 0.0035
маскирование     0.8078 ± 0.0031
всё вместе       0.8133 ± 0.0034
